# SmartPlate – NLP / RAG Pipeline

**Project:** SmartPlate – AI Nutrition Assistant  
**Block:** NLP / RAG (3/3)  
**Goal:** Build a Retrieval-Augmented Generation pipeline that answers nutrition questions based on WHO/DGE/Harvard guidelines.

## Pipeline

1. Load PDFs from knowledge base
2. Extract text → split into chunks
3. Generate embeddings (sentence-transformers)
4. Store in ChromaDB
5. Retrieval test
6. LLM (OpenAI gpt-4o-mini) generates answer with retrieved context
7. Compare 2 prompt strategies (Basic vs Few-Shot)
8. Evaluation on test questions

In [1]:
!pip install -q chromadb sentence-transformers pypdf openai langchain-text-splitters tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [2]:
# Clone das Repo, damit wir Zugriff auf die PDFs haben
import os, subprocess

if not os.path.exists("/content/smartplate"):
    !git clone https://github.com/Gianone-byte/smartplate.git /content/smartplate
else:
    !cd /content/smartplate && git pull

# Check
!ls -lh /content/smartplate/data/knowledge_base/

Cloning into '/content/smartplate'...
fatal: could not read Username for 'https://github.com': No such device or address
ls: cannot access '/content/smartplate/data/knowledge_base/': No such file or directory


In [9]:
import os
from pathlib import Path
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import pandas as pd
import numpy as np
from openai import OpenAI

print("Imports done")

Imports done


In [10]:
import os

# Check ob Repo schon da ist
if not os.path.exists("/content/smartplate"):
    print("Cloning repo...")
    !git clone https://github.com/Gianone-byte/smartplate.git /content/smartplate
else:
    print("Repo already exists, pulling latest...")
    !cd /content/smartplate && git pull

# Verifiziere PDFs
print("\n--- Content of knowledge_base/ ---")
!ls -lh /content/smartplate/data/knowledge_base/

Cloning repo...
Cloning into '/content/smartplate'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 69 (delta 10), reused 67 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 9.01 MiB | 25.27 MiB/s, done.
Resolving deltas: 100% (10/10), done.

--- Content of knowledge_base/ ---
total 1.3M
-rw-r--r-- 1 root root 778K May 25 11:28 dge_10_regeln.pdf
-rw-r--r-- 1 root root 359K May 25 11:28 harvard_healthy_eating.pdf
-rw-r--r-- 1 root root 119K May 25 11:28 who_healthy_diet.pdf


In [11]:
from google.colab import userdata
import os

# Read from Colab Secrets (das Schlüssel-Symbol links)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Test: Init client
client = OpenAI(api_key=OPENAI_API_KEY)

# Quick API-Test
test_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'API works' in 3 words"}],
    max_tokens=10,
)
print(f"OpenAI Test: {test_response.choices[0].message.content}")
print(f"Tokens used: {test_response.usage.total_tokens}")

OpenAI Test: API is functioning.
Tokens used: 20


In [12]:
KNOWLEDGE_BASE_DIR = "/content/smartplate/data/knowledge_base"

def load_pdf_text(pdf_path):
    """Extract all text from a PDF."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append({
                "page": i + 1,
                "text": text.strip(),
            })
    return pages


pdf_files = list(Path(KNOWLEDGE_BASE_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs:\n")

all_documents = []
for pdf in sorted(pdf_files):
    pages = load_pdf_text(pdf)
    total_chars = sum(len(p["text"]) for p in pages)
    print(f"  {pdf.name:35s} → {len(pages):2d} pages, {total_chars:,} chars")

    for page in pages:
        all_documents.append({
            "source": pdf.name,
            "page": page["page"],
            "text": page["text"],
        })

print(f"\n✅ Total document chunks (pages): {len(all_documents)}")

Found 3 PDFs:



  dge_10_regeln.pdf                   →  8 pages, 4,966 chars
  harvard_healthy_eating.pdf          →  6 pages, 5,306 chars
  who_healthy_diet.pdf                → 13 pages, 18,507 chars

✅ Total document chunks (pages): 27


In [13]:
# Split documents into chunks for embedding
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # ~500 chars per chunk
    chunk_overlap=100,       # 100 chars overlap zwischen chunks
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in all_documents:
    text_chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(text_chunks):
        chunks.append({
            "id": f"{doc['source']}_p{doc['page']}_c{i}",
            "source": doc["source"],
            "page": doc["page"],
            "chunk_idx": i,
            "text": chunk,
        })

print(f"Total chunks: {len(chunks)}")
print(f"\nFirst chunk preview:")
print("---")
print(chunks[0]["text"][:300])
print("---")

# Statistics
chunk_lengths = [len(c["text"]) for c in chunks]
print(f"\nChunk stats:")
print(f"  Min: {min(chunk_lengths)} chars")
print(f"  Max: {max(chunk_lengths)} chars")
print(f"  Mean: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")

Total chunks: 80

First chunk preview:
---
Gut essen und trinken – die DGE-
Empfehlungen
Bunt und gesund essen und dabei die Umwelt schonen, das sind die
DGE-Empfehlungen. Wer sich überwiegend von Obst und Gemüse,
Vollkorngetreide, Hülsenfrüchten sowie Nüssen und pﬂanzlichen Ölen
ernährt, schützt nicht nur seine Gesundheit, sondern schont da
---

Chunk stats:
  Min: 110 chars
  Max: 497 chars
  Mean: 411 chars


In [14]:
print("Loading sentence-transformer model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Model loaded, embedding dim: {embed_model.get_sentence_embedding_dimension()}")

print("\nGenerating embeddings for all chunks...")
texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=32)

print(f"\n Embeddings shape: {embeddings.shape}")
print(f"   {len(chunks)} chunks × {embeddings.shape[1]} dimensions")

Loading sentence-transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded, embedding dim: 384

Generating embeddings for all chunks...


/tmp/ipykernel_539/2239550147.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded, embedding dim: {embed_model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


 Embeddings shape: (80, 384)
   80 chunks × 384 dimensions


In [15]:
# Create ChromaDB collection
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# Delete existing collection if any
try:
    chroma_client.delete_collection("smartplate_nutrition")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="smartplate_nutrition",
    metadata={"description": "WHO + DGE + Harvard nutrition guidelines"},
)

# Add chunks to collection
collection.add(
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in chunks],
    metadatas=[
        {"source": c["source"], "page": c["page"], "chunk_idx": c["chunk_idx"]}
        for c in chunks
    ],
    ids=[c["id"] for c in chunks],
)

print(f"ChromaDB collection created with {collection.count()} chunks")

ChromaDB collection created with 80 chunks


In [16]:
def retrieve(query, top_k=3):
    """Retrieve top-k most similar chunks for a query."""
    query_emb = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_emb,
        n_results=top_k,
    )
    return results

# Test mit einer Beispiel-Frage
test_query = "How much sugar should I eat per day?"
results = retrieve(test_query)

print(f"Query: {test_query}\n")
print("Top 3 results:\n")
for i, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0]), 1):
    print(f"--- Result {i} (distance: {dist:.4f}) ---")
    print(f"Source: {meta['source']} (page {meta['page']})")
    print(f"Text: {doc[:300]}...")
    print()

Query: How much sugar should I eat per day?

Top 3 results:

--- Result 1 (distance: 0.5686) ---
Source: who_healthy_diet.pdf (page 4)
Text: 9 years of age, respectively.
Sugars
The consumption of free sugars should be
limited to less than 10% of total daily energy
intake, which is equivalent to 50 g (or about
12 level teaspoons) for a person of healthy
body weight consuming about 2000 calories
per day. Limiting further to 5% or less of ...

--- Result 2 (distance: 0.6479) ---
Source: who_healthy_diet.pdf (page 4)
Text: aim for at least 400 grams of fruits and
vegetables per day, with lesser amounts for
children under 10: at least 250 or 350 grams
for children 2–5 or 6–9 years of age,
respectively.
Everyone older than 10 years of age should also
aim for a daily intake of at least 25 grams of
naturally-occurring die...

--- Result 3 (distance: 0.9529) ---
Source: who_healthy_diet.pdf (page 3)
Text: amounts of free sugars, the consumption of
which should be limited.
Everyone older than 1

In [17]:
def rag_answer_basic(query, top_k=3, model="gpt-4o-mini"):
    """Basic RAG: retrieve context + simple prompt."""
    # 1. Retrieve
    results = retrieve(query, top_k=top_k)
    context_chunks = results['documents'][0]
    sources = results['metadatas'][0]

    # 2. Build context
    context = "\n\n".join([
        f"[Source: {s['source']}, page {s['page']}]\n{chunk}"
        for chunk, s in zip(context_chunks, sources)
    ])

    # 3. Basic Prompt
    system_prompt = "You are a nutrition assistant. Answer based only on the provided context."
    user_prompt = f"""Context:
{context}

Question: {query}

Answer in 2-3 sentences."""

    # 4. LLM call
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=200,
        temperature=0.3,
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [f"{s['source']} (p.{s['page']})" for s in sources],
        "tokens": response.usage.total_tokens,
    }

# Test
test_question = "How much sugar should I eat per day?"
result = rag_answer_basic(test_question)
print(f"Q: {test_question}\n")
print(f"A: {result['answer']}\n")
print(f"Sources: {result['sources']}")
print(f"Tokens used: {result['tokens']}")

Q: How much sugar should I eat per day?

A: You should limit your consumption of free sugars to less than 10% of your total daily energy intake, which is about 50 grams (or 12 level teaspoons) for a person consuming around 2000 calories per day. For additional health benefits, it is recommended to limit free sugars to 5% or less of total daily energy intake.

Sources: ['who_healthy_diet.pdf (p.4)', 'who_healthy_diet.pdf (p.4)', 'who_healthy_diet.pdf (p.3)']
Tokens used: 432


In [18]:
def rag_answer_smartplate(food_class, kcal, health_label, user_question=None, top_k=3, model="gpt-4o-mini"):
    """Advanced RAG: integrate CV + ML output + nutrition KB."""
    # Default question if none provided
    if user_question is None:
        user_question = f"Is {food_class.replace('_', ' ')} a healthy choice and what should I know about it?"

    # 1. Retrieve relevant chunks
    results = retrieve(user_question, top_k=top_k)
    context_chunks = results['documents'][0]
    sources = results['metadatas'][0]

    context = "\n\n".join([
        f"[Source: {s['source']}, page {s['page']}]\n{chunk}"
        for chunk, s in zip(context_chunks, sources)
    ])

    # 2. Advanced Prompt mit Pipeline-Context
    system_prompt = """You are SmartPlate, a friendly AI nutrition coach.
Your goal: Give helpful, evidence-based nutrition advice — not diet shaming.

Guidelines:
- Use the provided nutrition guidelines as authoritative source
- Be balanced: acknowledge enjoyment, explain trade-offs
- Avoid dramatic language like "bad" or "forbidden" food
- Suggest practical alternatives or balance tips when relevant
- Mention sources when citing specific values
"""

    user_prompt = f"""The user has just shown a photo of: **{food_class.replace('_', ' ')}**

Computer vision classification: {food_class}
Estimated calories: {kcal:.0f} kcal per 100g
Health category (from our classifier): {health_label}

Relevant nutrition guidelines:
{context}

User's question: {user_question}

Please respond in 3-5 sentences. Be helpful, factual, and friendly."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=300,
        temperature=0.5,
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [f"{s['source']} (p.{s['page']})" for s in sources],
        "tokens": response.usage.total_tokens,
    }

# Test 1: Pizza
print("="*60)
print("TEST 1: Pizza")
print("="*60)
result = rag_answer_smartplate("pizza", kcal=266, health_label="unhealthy")
print(f"\n{result['answer']}\n")
print(f"Sources: {result['sources']}")
print(f"Tokens: {result['tokens']}\n")

# Test 2: Greek salad
print("="*60)
print("TEST 2: Greek Salad")
print("="*60)
result = rag_answer_smartplate("greek_salad", kcal=107, health_label="healthy")
print(f"\n{result['answer']}\n")
print(f"Sources: {result['sources']}")
print(f"Tokens: {result['tokens']}\n")

# Test 3: Donuts mit Custom-Frage
print("="*60)
print("TEST 3: Donuts - 'Can I still eat this if I'm on a diet?'")
print("="*60)
result = rag_answer_smartplate(
    "donuts", kcal=452, health_label="unhealthy",
    user_question="Can I still eat this if I'm on a diet?",
)
print(f"\n{result['answer']}\n")
print(f"Sources: {result['sources']}")
print(f"Tokens: {result['tokens']}")

TEST 1: Pizza

Pizza can be a delicious treat, but it's important to consider its nutritional content. With an estimated 266 kcal per 100g, it can be higher in calories, especially if loaded with cheese and processed meats. While it may not fit into the "healthy" category, you can still enjoy it in moderation. To make it a more balanced choice, consider opting for whole grain crust, adding plenty of vegetables as toppings, and choosing lean proteins like chicken or plant-based options. Balancing pizza with a side salad or veggies can help enhance the nutritional value of your meal!

Sources: ['harvard_healthy_eating.pdf (p.4)', 'who_healthy_diet.pdf (p.9)', 'harvard_healthy_eating.pdf (p.3)']
Tokens: 669

TEST 2: Greek Salad

Greek salad is indeed a healthy choice! It's packed with a variety of vegetables, which provide essential vitamins and minerals, and it's low in calories, at about 107 kcal per 100g. The inclusion of ingredients like olives and feta cheese adds healthy fats and pr

In [19]:
# Test-Fragen für systematische Evaluation
TEST_QUESTIONS = [
    {
        "id": "q1",
        "question": "How much sugar should I eat per day?",
        "expected_keywords": ["10%", "50g", "sugar", "free"],
        "expected_source": "who",
    },
    {
        "id": "q2",
        "question": "Are saturated fats unhealthy?",
        "expected_keywords": ["saturated", "fat", "10%", "energy"],
        "expected_source": "who",
    },
    {
        "id": "q3",
        "question": "How much salt is recommended daily?",
        "expected_keywords": ["salt", "5g", "sodium"],
        "expected_source": "who",
    },
    {
        "id": "q4",
        "question": "What is a balanced meal?",
        "expected_keywords": ["vegetables", "fruits", "whole grains", "protein"],
        "expected_source": "harvard",
    },
    {
        "id": "q5",
        "question": "Why are fruits and vegetables important?",
        "expected_keywords": ["fruits", "vegetables", "400g", "fiber"],
        "expected_source": "who",
    },
    {
        "id": "q6",
        "question": "Should I drink milk?",
        "expected_keywords": ["dairy", "milk", "water"],
        "expected_source": "harvard",
    },
    {
        "id": "q7",
        "question": "Was ist eine ausgewogene Ernährung?",  # German query
        "expected_keywords": ["Lebensmittel", "Vielfalt"],
        "expected_source": "dge",
    },
    {
        "id": "q8",
        "question": "Are eggs healthy?",
        "expected_keywords": ["protein", "eggs", "diet"],
        "expected_source": "any",
    },
]

print(f"Total test questions: {len(TEST_QUESTIONS)}")

Total test questions: 8


In [20]:
def rag_answer_strategy_1(query, top_k=3, model="gpt-4o-mini"):
    """Strategy 1: Basic — minimal context, simple instruction."""
    results = retrieve(query, top_k=top_k)
    context = "\n\n".join(results['documents'][0])

    prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer briefly."

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0.3,
    )
    return {
        "answer": response.choices[0].message.content,
        "sources": [m for m in results['metadatas'][0]],
        "tokens": response.usage.total_tokens,
    }


def rag_answer_strategy_2(query, top_k=3, model="gpt-4o-mini"):
    """Strategy 2: Structured XML + role + grounding instruction."""
    results = retrieve(query, top_k=top_k)

    context_xml = ""
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
        context_xml += f"<source_{i} file='{meta['source']}' page='{meta['page']}'>\n{doc}\n</source_{i}>\n\n"

    system_prompt = """You are a nutrition expert assistant.
- Answer ONLY based on the provided sources
- If the sources don't contain the answer, say "I don't have that information in my sources"
- Cite source numbers (e.g. "according to source_1")
- Be precise with numbers"""

    user_prompt = f"""<sources>
{context_xml}</sources>

<question>{query}</question>

Please answer in 2-4 sentences."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=250,
        temperature=0.3,
    )
    return {
        "answer": response.choices[0].message.content,
        "sources": [m for m in results['metadatas'][0]],
        "tokens": response.usage.total_tokens,
    }


# Test mit einer Frage
print("="*60)
print("STRATEGY COMPARISON: 'How much sugar per day?'")
print("="*60)

s1 = rag_answer_strategy_1("How much sugar should I eat per day?")
print("\n--- Strategy 1 (Basic) ---")
print(s1['answer'])
print(f"Tokens: {s1['tokens']}")

s2 = rag_answer_strategy_2("How much sugar should I eat per day?")
print("\n--- Strategy 2 (Structured XML) ---")
print(s2['answer'])
print(f"Tokens: {s2['tokens']}")

STRATEGY COMPARISON: 'How much sugar per day?'

--- Strategy 1 (Basic) ---
You should limit your intake of free sugars to less than 10% of your total daily energy intake, which is about 50 grams (or 12 teaspoons) for a person consuming 2000 calories per day. For additional health benefits, aim for 5% or less. Children under 10 should have lesser amounts based on their age.
Tokens: 367

--- Strategy 2 (Structured XML) ---
The consumption of free sugars should be limited to less than 10% of total daily energy intake, which is equivalent to 50 grams (about 12 level teaspoons) for a person consuming around 2000 calories per day. For additional health benefits, it is suggested to limit free sugars to 5% or less of total daily energy intake. This recommendation applies throughout the life course (according to source_1).
Tokens: 521


In [21]:
def evaluate_strategy(strategy_func, strategy_name, test_questions):
    """Evaluate a RAG strategy on test questions."""
    results = []

    for q in test_questions:
        try:
            answer = strategy_func(q["question"])

            # Check keyword coverage
            answer_lower = answer["answer"].lower()
            keywords_found = sum(1 for kw in q["expected_keywords"] if kw.lower() in answer_lower)
            keyword_coverage = keywords_found / len(q["expected_keywords"])

            # Check source relevance
            sources_str = " ".join([s["source"].lower() for s in answer["sources"]])
            source_match = (
                q["expected_source"] == "any" or
                q["expected_source"] in sources_str
            )

            results.append({
                "question_id": q["id"],
                "question": q["question"][:50] + "...",
                "answer": answer["answer"][:120] + "...",
                "keyword_coverage": keyword_coverage,
                "source_match": source_match,
                "tokens": answer["tokens"],
                "strategy": strategy_name,
            })
        except Exception as e:
            print(f"Error on {q['id']}: {e}")
            results.append({
                "question_id": q["id"],
                "strategy": strategy_name,
                "error": str(e),
                "keyword_coverage": 0,
                "source_match": False,
                "tokens": 0,
            })

    return pd.DataFrame(results)


print("Evaluating Strategy 1 (Basic)...")
df_s1 = evaluate_strategy(rag_answer_strategy_1, "Strategy 1 (Basic)", TEST_QUESTIONS)

print("\nEvaluating Strategy 2 (Structured XML)...")
df_s2 = evaluate_strategy(rag_answer_strategy_2, "Strategy 2 (Structured XML)", TEST_QUESTIONS)

# Combine
df_eval = pd.concat([df_s1, df_s2], ignore_index=True)
print("\n Evaluation done")

Evaluating Strategy 1 (Basic)...

Evaluating Strategy 2 (Structured XML)...

✅ Evaluation done


In [22]:
# Aggregate per strategy
summary = df_eval.groupby("strategy").agg({
    "keyword_coverage": "mean",
    "source_match": "mean",
    "tokens": "mean",
}).round(3)
summary.columns = ["Avg Keyword Coverage", "Source Match Rate", "Avg Tokens"]

print("=== Strategy Comparison ===\n")
display(summary)

# Detail table
print("\n=== Per-Question Results ===\n")
display(df_eval[["strategy", "question_id", "keyword_coverage", "source_match", "tokens"]].pivot_table(
    index="question_id", columns="strategy", values=["keyword_coverage", "source_match", "tokens"]
).round(2))

# Save
df_eval.to_csv("/content/rag_evaluation.csv", index=False)
summary.to_csv("/content/rag_strategy_comparison.csv")
print("\n Files saved")

=== Strategy Comparison ===



,Avg Keyword Coverage,Source Match Rate,Avg Tokens
strategy,,,
Strategy 1 (Basic),0.594,0.875,356.00
Strategy 2 (Structured XML),0.365,0.875,493.75



=== Per-Question Results ===



keyword_coverage                                   source_match  \
strategy    Strategy 1 (Basic) Strategy 2 (Structured XML) Strategy 1 (Basic)   
question_id                                                                     
q1                        0.75                        0.75                1.0   
q2                        0.50                        0.50                1.0   
q3                        0.67                        0.67                1.0   
q4                        0.25                        0.00                1.0   
q5                        0.75                        0.50                1.0   
q6                        0.33                        0.00                0.0   
q7                        0.50                        0.50                1.0   
q8                        1.00                        0.00                1.0   

                                                    tokens  \
strategy    Strategy 2 (Structured XML) Strategy 1 (Basic)   
question_id                                                  
q1                                  1.0              388.0   
q2                                  1.0              285.0   
q3                                  1.0              375.0   
q4                                  1.0              284.0   
q5                                  1.0              360.0   
q6                                  0.0              339.0   
q7                                  1.0              439.0   
q8                                  1.0              378.0   

                                         
strategy    Strategy 2 (Structured XML)  
question_id                              
q1                                513.0  
q2                                453.0  
q3                                549.0  
q4                                423.0  
q5                                515.0  
q6                                438.0  
q7                                572.0  
q8                                487.0


✅ Files saved


In [23]:
import pickle

# Best strategy = Strategy 1 (Basic)
BEST_STRATEGY = "Strategy 1 (Basic)"

# Save metadata about the RAG setup
rag_config = {
    "best_strategy": BEST_STRATEGY,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "llm_model": "gpt-4o-mini",
    "chunk_size": 500,
    "chunk_overlap": 100,
    "top_k": 3,
    "knowledge_sources": ["who_healthy_diet.pdf", "dge_10_regeln.pdf", "harvard_healthy_eating.pdf"],
    "total_chunks": len(chunks),
    "evaluation_summary": {
        "strategy_1_keyword_coverage": float(df_s1["keyword_coverage"].mean()),
        "strategy_2_keyword_coverage": float(df_s2["keyword_coverage"].mean()),
        "strategy_1_avg_tokens": float(df_s1["tokens"].mean()),
        "strategy_2_avg_tokens": float(df_s2["tokens"].mean()),
    },
}

# Save chunks list (for later rebuild of ChromaDB without re-parsing PDFs)
with open("/content/rag_chunks.json", "w") as f:
    import json
    json.dump(chunks, f, indent=2)

with open("/content/rag_config.json", "w") as f:
    json.dump(rag_config, f, indent=2)

print("RAG configuration saved:")
print(json.dumps(rag_config, indent=2))

RAG configuration saved:
{
  "best_strategy": "Strategy 1 (Basic)",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "llm_model": "gpt-4o-mini",
  "chunk_size": 500,
  "chunk_overlap": 100,
  "top_k": 3,
  "knowledge_sources": [
    "who_healthy_diet.pdf",
    "dge_10_regeln.pdf",
    "harvard_healthy_eating.pdf"
  ],
  "total_chunks": 80,
  "evaluation_summary": {
    "strategy_1_keyword_coverage": 0.59375,
    "strategy_2_keyword_coverage": 0.3645833333333333,
    "strategy_1_avg_tokens": 356.0,
    "strategy_2_avg_tokens": 493.75
  }
}


In [24]:
def smartplate_full_pipeline(food_class, kcal, health_label, user_question=None):
    """Simulates the full SmartPlate pipeline:
    CV (food_class) → ML (kcal + health_label) → NLP (natural language answer)
    """
    if user_question is None:
        user_question = f"Tell me about {food_class.replace('_', ' ')} - is it healthy?"

    # Retrieve
    results = retrieve(user_question, top_k=3)
    context = "\n\n".join([
        f"[Source: {m['source']}, page {m['page']}]\n{doc}"
        for doc, m in zip(results['documents'][0], results['metadatas'][0])
    ])

    # SmartPlate Production Prompt (best of both)
    system_prompt = """You are SmartPlate, a friendly AI nutrition coach.
- Be evidence-based: use the provided context
- Be balanced: acknowledge enjoyment, explain trade-offs
- Avoid moralizing language ("bad", "forbidden")
- Suggest practical alternatives when appropriate
- Be concise: 3-5 sentences max"""

    user_prompt = f"""The user uploaded a photo of: **{food_class.replace('_', ' ')}**

📷 Vision identifies: {food_class}
🍽 Nutrition (per 100g): ~{kcal:.0f} kcal
🏷  Health classification: {health_label}

Reference guidelines:
{context}

User asks: {user_question}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=300,
        temperature=0.5,
    )

    return {
        "food_class": food_class,
        "kcal": kcal,
        "health_label": health_label,
        "answer": response.choices[0].message.content,
        "sources": [f"{m['source']} (p.{m['page']})" for m in results['metadatas'][0]],
        "tokens": response.usage.total_tokens,
    }


# Test with 3 different scenarios
print("="*70)
print("SMARTPLATE FULL PIPELINE - Integration Test")
print("="*70)

scenarios = [
    ("pizza", 266, "unhealthy", None),
    ("greek_salad", 107, "healthy", "Is this enough protein for lunch?"),
    ("ice_cream", 207, "unhealthy", "I'm sad and want comfort food - is this okay?"),
]

for food, kcal, health, question in scenarios:
    print(f"\n{'='*70}")
    print(f"INPUT: {food} | {kcal} kcal | {health}")
    if question:
        print(f"Q: {question}")
    print(f"{'='*70}")
    result = smartplate_full_pipeline(food, kcal, health, question)
    print(f"\n{result['answer']}\n")
    print(f"Sources: {result['sources']}")
    print(f"Tokens: {result['tokens']}")

SMARTPLATE FULL PIPELINE - Integration Test

INPUT: pizza | 266 kcal | unhealthy

Pizza can be a delicious treat, but its healthiness often depends on the ingredients and portion size. While it provides carbohydrates and some protein, it can also be high in calories, saturated fats, and sodium, especially if it's loaded with cheese and processed meats. To make it a healthier option, consider choosing whole grain crust, adding plenty of vegetables, and opting for lean proteins like chicken or plant-based toppings. Enjoying pizza in moderation, along with a balanced diet, can be part of a healthy lifestyle!

Sources: ['harvard_healthy_eating.pdf (p.4)', 'harvard_healthy_eating.pdf (p.3)', 'harvard_healthy_eating.pdf (p.3)']
Tokens: 585

INPUT: greek_salad | 107 kcal | healthy
Q: Is this enough protein for lunch?

A Greek salad is a nutritious choice, typically low in calories and rich in vitamins, minerals, and healthy fats from ingredients like olives and feta cheese. However, it may no

In [25]:
from google.colab import files

files.download("/content/rag_evaluation.csv")
files.download("/content/rag_strategy_comparison.csv")
files.download("/content/rag_chunks.json")
files.download("/content/rag_config.json")

print("Files downloaded - move to assets/screenshots/rag/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded - move to assets/screenshots/rag/
